# LLM Training Lab, lltTrainingLoop: learning-rate sweep on a 10.77M char model (T4)

FROM SCRATCH. Char-level Tiny Shakespeare. Learning rates 1e-4, 3e-4, 1e-3, 3e-3, 1e-2 with 3 seeds each, then extra seeds at the best rate. fp16 autocast + GradScaler, dtype picked by compute capability. One run per GPU on Kaggle's 2x T4.

In [ ]:

# Runtime check. It prints the fix for each problem it finds.
import platform, shutil, sys, time
import torch
START = time.time()
print("Python", platform.python_version(), "| torch", torch.__version__)
if not torch.cuda.is_available():
    print("FIX: no GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
else:
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i), "capability", torch.cuda.get_device_capability(i))
    if torch.cuda.get_device_capability(0) < (7, 0):
        print("FIX: this GPU is older than the T4 and current PyTorch wheels may not support it. Choose a T4.")
    print("torch.cuda.is_bf16_supported() =", torch.cuda.is_bf16_supported(), "(True on a T4 too, so do not use it to pick the dtype)")
print("Free disk", round(shutil.disk_usage(".").free / 1e9, 1), "GB")


In [ ]:
LIB = 'import os\nimport urllib.request\nimport numpy as np\nimport torch\n\nURL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"\n\n\ndef prepare(data_dir="data"):\n    # Download once, map each character to an id, write uint8 token files.\n    os.makedirs(data_dir, exist_ok=True)\n    path = os.path.join(data_dir, "input.txt")\n    if not os.path.exists(path):\n        urllib.request.urlretrieve(URL, path)\n    text = open(path, encoding="utf-8").read()\n    chars = sorted(set(text))\n    stoi = {c: i for i, c in enumerate(chars)}\n    if not os.path.exists(os.path.join(data_dir, "val.bin")):\n        ids = np.array([stoi[c] for c in text], dtype=np.uint8)\n        n = int(0.9 * len(ids))\n        ids[:n].tofile(os.path.join(data_dir, "train.bin"))\n        ids[n:].tofile(os.path.join(data_dir, "val.bin"))\n    return chars\n\n\nclass Sampler:\n    # Seeded batch order over a memmapped token file. Its position is two integers.\n    def __init__(self, path, block, batch, seed):\n        self.data = np.memmap(path, dtype=np.uint8, mode="r")\n        self.block, self.batch, self.seed = block, batch, seed\n        self.n = (len(self.data) - 1 - block) // block\n        self.epoch, self.pos = 0, 0\n        self._shuffle()\n\n    def _shuffle(self):\n        # The order depends only on (seed, epoch), so it can be rebuilt after a restart.\n        rng = np.random.default_rng([self.seed, self.epoch])\n        offset = int(rng.integers(self.block))\n        self.starts = offset + self.block * rng.permutation(self.n)\n\n    def next_batch(self):\n        if self.pos + self.batch > self.n:\n            self.epoch, self.pos = self.epoch + 1, 0\n            self._shuffle()\n        s = self.starts[self.pos:self.pos + self.batch]\n        self.pos += self.batch\n        x = np.stack([self.data[i:i + self.block] for i in s]).astype(np.int64)\n        y = np.stack([self.data[i + 1:i + 1 + self.block] for i in s]).astype(np.int64)\n        return torch.from_numpy(x), torch.from_numpy(y)\n\n    def state_dict(self):\n        return {"seed": self.seed, "epoch": self.epoch, "pos": self.pos}\n\n    def load_state_dict(self, state):\n        assert state["seed"] == self.seed, "sampler seed differs from the checkpoint"\n        self.epoch, self.pos = state["epoch"], state["pos"]\n        self._shuffle()\n\n\nimport math\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass Block(nn.Module):\n    def __init__(self, d, n_head, dropout):\n        super().__init__()\n        self.n_head, self.dropout = n_head, dropout\n        self.ln1, self.ln2 = nn.LayerNorm(d, bias=False), nn.LayerNorm(d, bias=False)\n        self.qkv, self.proj = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)\n        self.fc, self.out = nn.Linear(d, 4 * d, bias=False), nn.Linear(4 * d, d, bias=False)\n        self.drop = nn.Dropout(dropout)\n\n    def forward(self, x):\n        B, T, C = x.shape\n        q, k, v = self.qkv(self.ln1(x)).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))\n        p = self.dropout if self.training else 0.0\n        a = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)\n        x = x + self.drop(self.proj(a.transpose(1, 2).reshape(B, T, C)))\n        return x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))\n\n\nclass GPT(nn.Module):\n    def __init__(self, vocab, block, n_layer, n_head, d, dropout=0.0):\n        super().__init__()\n        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(block, d)\n        self.drop = nn.Dropout(dropout)\n        self.blocks = nn.ModuleList([Block(d, n_head, dropout) for _ in range(n_layer)])\n        self.ln_f, self.head = nn.LayerNorm(d, bias=False), nn.Linear(d, vocab, bias=False)\n        for name, p in self.named_parameters():\n            if p.dim() == 2:\n                # GPT-2 init: std 0.02, and the residual output layers scaled by 1/sqrt(2L)\n                std = 0.02 / math.sqrt(2 * n_layer) if name.endswith(("proj.weight", "out.weight")) else 0.02\n                nn.init.normal_(p, 0.0, std)\n\n    def forward(self, idx):\n        x = self.drop(self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device)))\n        for blk in self.blocks:\n            x = blk(x)\n        return self.head(self.ln_f(x))\n\n\ndef param_groups(model, weight_decay):\n    # Decay the weight matrices of linear layers only. Norm gains and embeddings keep their size.\n    decay, no_decay = [], []\n    for module in model.modules():\n        for name, p in module.named_parameters(recurse=False):\n            if isinstance(module, nn.Linear) and name == "weight":\n                decay.append(p)\n            else:\n                no_decay.append(p)\n    return [{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}]\n\n\nfrom torch.optim.lr_scheduler import LambdaLR\n\n\ndef lr_cosine(step, peak, warmup, total, min_ratio=0.1):\n    if step < warmup:\n        return peak * (step + 1) / warmup\n    progress = min(1.0, (step - warmup) / max(1, total - warmup))\n    return peak * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress)))\n\n\ndef lr_wsd(step, peak, warmup, total, min_ratio=0.1, decay_frac=0.2):\n    # Warmup, then a flat stable phase, then a linear decay over the last decay_frac of steps.\n    decay_start = int(total * (1 - decay_frac))\n    if step < warmup:\n        return peak * (step + 1) / warmup\n    if step < decay_start:\n        return peak\n    progress = min(1.0, (step - decay_start) / max(1, total - decay_start))\n    return peak * (1 - (1 - min_ratio) * progress)\n\n\ndef make_scheduler(opt, cfg):\n    fn = lr_wsd if cfg["schedule"] == "wsd" else lr_cosine\n    # LambdaLR multiplies the optimizer\'s base lr (cfg["lr"]) by this factor at every step.\n    return LambdaLR(opt, lambda s: fn(s, cfg["lr"], cfg["warmup"], cfg["steps"]) / cfg["lr"])\n\n\nimport random\n\n\ndef save_checkpoint(path, model, opt, sched, scaler, sampler, step, cfg):\n    state = {\n        "model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),\n        "scaler": scaler.state_dict(), "sampler": sampler.state_dict(), "step": step, "cfg": cfg,\n        "chars": chars,\n        "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},\n    }\n    torch.save(state, path + ".tmp")\n    os.replace(path + ".tmp", path)  # a session that dies mid-save leaves the old file intact\n\n\ndef load_checkpoint(path, model, opt, sched, scaler, sampler, skip=()):\n    # weights_only=False because the RNG states are Python and NumPy objects. Load only your own files.\n    state = torch.load(path, map_location="cpu", weights_only=False)\n    model.load_state_dict(state["model"])\n    if "optimizer" not in skip:\n        opt.load_state_dict(state["optimizer"])\n    if "scheduler" not in skip:\n        sched.load_state_dict(state["scheduler"])\n    scaler.load_state_dict(state["scaler"])\n    if "sampler" not in skip:\n        sampler.load_state_dict(state["sampler"])\n    if "rng" not in skip:\n        random.setstate(state["rng"]["python"])\n        np.random.set_state(state["rng"]["numpy"])\n        torch.set_rng_state(state["rng"]["torch"])\n        if state["rng"]["cuda"] is not None and torch.cuda.is_available():\n            torch.cuda.set_rng_state_all(state["rng"]["cuda"])\n    return state["step"]\n\n\nimport json\nimport time\nfrom contextlib import nullcontext\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef pick_dtype():\n    # T4 is compute capability 7.5: fp16 tensor cores, no bf16 ones, although\n    # torch.cuda.is_bf16_supported() returns True there. Decide by capability.\n    if device.type != "cuda":\n        return torch.float32\n    return torch.bfloat16 if torch.cuda.get_device_capability() >= (8, 0) else torch.float16\n\n\n@torch.no_grad()\ndef evaluate(model, cfg, dtype):\n    # Mean loss over every window of the validation file, in a fixed order.\n    model.eval()\n    val = np.memmap("data/val.bin", dtype=np.uint8, mode="r")\n    T = cfg["block"]\n    starts = np.arange(0, len(val) - T - 1, T)\n    total, count = 0.0, 0\n    for i in range(0, len(starts), 64):\n        s = starts[i:i + 64]\n        x = torch.from_numpy(np.stack([val[j:j + T] for j in s]).astype(np.int64)).to(device)\n        y = torch.from_numpy(np.stack([val[j + 1:j + 1 + T] for j in s]).astype(np.int64)).to(device)\n        with autocast(dtype):\n            logits = model(x)\n        total += F.cross_entropy(logits.float().flatten(0, 1), y.flatten(), reduction="sum").item()\n        count += y.numel()\n    model.train()\n    return total / count\n\n\ndef autocast(dtype):\n    return torch.autocast(device.type, dtype=dtype) if dtype != torch.float32 else nullcontext()\n\n\ndef train(cfg, log_path=None):\n    torch.manual_seed(cfg["seed"])\n    dtype = pick_dtype()\n    model = GPT(len(chars), cfg["block"], cfg["n_layer"], cfg["n_head"], cfg["d"], cfg["dropout"]).to(device)\n    opt = torch.optim.AdamW(param_groups(model, cfg["wd"]), lr=cfg["lr"], betas=(0.9, 0.99))\n    sched = make_scheduler(opt, cfg)\n    scaler = torch.amp.GradScaler(device.type, enabled=(dtype == torch.float16))\n    sampler = Sampler("data/train.bin", cfg["block"], cfg["micro_batch"], cfg["seed"])\n    step = 0\n    if cfg.get("resume"):\n        step = load_checkpoint(cfg["ckpt"], model, opt, sched, scaler, sampler, skip=cfg.get("skip", ()))\n    elif log_path:\n        open(log_path, "w").close()\n    N, history, t0 = cfg["accum"], [], time.time()\n    while step < cfg.get("stop_at", cfg["steps"]):\n        opt.zero_grad(set_to_none=True)\n        micro_losses = []\n        for _ in range(N):\n            x, y = (t.to(device) for t in sampler.next_batch())\n            with autocast(dtype):\n                loss = F.cross_entropy(model(x).float().flatten(0, 1), y.flatten())\n            scaler.scale(loss / N if cfg.get("divide_loss", True) else loss).backward()\n            micro_losses.append(loss.item())\n            if cfg.get("sched_per_micro"):\n                sched.step()                     # broken on purpose in step 10\n        scaler.unscale_(opt)                     # clip the true gradients, not the scaled ones\n        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"]).item()\n        lr = opt.param_groups[0]["lr"]\n        scaler.step(opt)                         # skips the update if fp16 gradients overflowed\n        scaler.update()\n        if not cfg.get("sched_per_micro"):\n            sched.step()\n        step += 1\n        row = {"step": step, "trainLoss": sum(micro_losses) / N, "gradNorm": norm, "lr": lr}\n        if step % cfg["eval_every"] == 0 or step == cfg["steps"]:\n            row["valLoss"] = evaluate(model, cfg, dtype)\n            row["elapsedSec"] = round(time.time() - t0, 1)\n        history.append(row)\n        if log_path:\n            with open(log_path, "a") as f:\n                f.write(json.dumps(row) + "\\n")\n        if cfg.get("ckpt") and step % cfg.get("ckpt_every", 10**9) == 0:\n            save_checkpoint(cfg["ckpt"], model, opt, sched, scaler, sampler, step, cfg)\n    return model, history\n\n\nchars = prepare()\n'
import os
WORK = os.environ.get('LLT_WORKDIR', '/kaggle/working')
open(os.path.join(WORK, 'lltloop.py'), 'w').write(LIB)
WORKER = '\nimport json, os, platform, sys, time\nWORK = os.environ.get("LLT_WORKDIR", "/kaggle/working")\nos.chdir(WORK)\nsys.path.insert(0, WORK)\nimport torch\nimport lltloop as L\nCUDA = torch.cuda.is_available()\njobs = json.load(open(sys.argv[1]))\n\ndef clean(x):\n    if isinstance(x, float) and (x != x or abs(x) == float("inf")):\n        return None\n    if isinstance(x, dict):\n        return {k: clean(v) for k, v in x.items()}\n    if isinstance(x, list):\n        return [clean(v) for v in x]\n    return x\n\nSTART, LIMIT = float(os.environ["LLT_START"]), float(os.environ["LLT_LIMIT_SEC"])\nfor job in jobs:\n    if time.time() - START > LIMIT:\n        print("time limit reached, skipping", job["runId"], flush=True)\n        continue\n    cfg = job["cfg"]\n    if CUDA:\n        torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    try:\n        model, hist = L.train(cfg)\n        status = "ok"\n    except Exception as e:\n        hist, status = [], "error: " + repr(e)\n    wall = time.time() - t0\n    tokens = cfg["steps"] * cfg["micro_batch"] * cfg["accum"] * cfg["block"]\n    nan_steps = sum(1 for r in hist if r["trainLoss"] != r["trainLoss"])\n    keep = [dict(r, tokens=r["step"] * cfg["micro_batch"] * cfg["accum"] * cfg["block"]) for r in hist\n            if r["step"] % 10 == 0 or "valLoss" in r or r["step"] <= 10]\n    vals = [r["valLoss"] for r in hist if "valLoss" in r]\n    finite = [v for v in vals if v == v and abs(v) != float("inf")]\n    rec = {\n        "runId": job["runId"], "module": "lltTrainingLoop", "tier": "T2", "label": job["label"],\n        "hardware": {"gpu": torch.cuda.get_device_name(0) if CUDA else None, "count": 1 if CUDA else 0, "vramGB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if CUDA else None,\n                     "capability": list(torch.cuda.get_device_capability()) if CUDA else None, "note": "Kaggle 2x T4 session, one run per GPU"},\n        "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "numpy": L.np.__version__,\n                "dtype": str(L.pick_dtype()), "bf16SupportedFlag": torch.cuda.is_bf16_supported() if CUDA else None},\n        "notebookCommit": None, "config": cfg, "seed": cfg["seed"], "status": status,\n        "params": sum(p.numel() for p in L.GPT(len(L.chars), cfg["block"], cfg["n_layer"], cfg["n_head"], cfg["d"]).parameters()),\n        "metrics": keep,\n        "evals": {"finalValLoss": vals[-1] if vals else None,\n                  "bestValLoss": min(finite) if finite else None,\n                  "nanTrainSteps": nan_steps},\n        "wallClockSec": round(wall, 1), "tokPerSecInclEval": round(tokens / wall) if hist else None,\n        "peakMemGB": round(torch.cuda.max_memory_allocated() / 1e9, 3) if CUDA else None,\n        "costUSD": 0, "priceCheckedOn": None, "date": time.strftime("%Y-%m-%d"),\n    }\n    os.makedirs("runs", exist_ok=True)\n    with open(os.path.join("runs", job["runId"] + ".json"), "w") as f:\n        json.dump(clean(rec), f, indent=1)\n    print(job["runId"], status, "final val", rec["evals"]["finalValLoss"], "wall", rec["wallClockSec"], "tok/s", rec["tokPerSecInclEval"], flush=True)\n'
open(os.path.join(WORK, 'worker.py'), 'w').write(WORKER)
print('wrote lltloop.py', len(LIB), 'chars and worker.py')

In [ ]:

import json, os, subprocess, sys
os.chdir(WORK)
sys.path.insert(0, WORK)
import lltloop as L
print("vocab", len(L.chars), "| device", L.device, "| dtype", L.pick_dtype())
T2 = dict(n_layer=6, n_head=6, d=384, block=256, dropout=0.2, micro_batch=64, accum=1, steps=2000,
          warmup=100, schedule="cosine", wd=0.1, clip=1.0, eval_every=250)
if os.environ.get("LLT_SMOKE"):  # local CPU check of this notebook only
    T2.update(n_layer=2, n_head=2, d=32, block=32, micro_batch=4, steps=20, eval_every=10)
LIMIT = float(os.environ.get("LLT_LIMIT_SEC", str(150 * 60)))

def run_pool(jobs, tag):
    n = max(1, torch.cuda.device_count())
    procs = []
    for g in range(n):
        mine = jobs[g::n]
        if not mine:
            continue
        path = f"jobs_{tag}_{g}.json"
        json.dump(mine, open(path, "w"))
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), LLT_START=str(START), LLT_LIMIT_SEC=str(LIMIT))
        procs.append(subprocess.Popen([sys.executable, "worker.py", path], env=env,
                                      stdout=open(f"worker_{tag}_{g}.log", "w"), stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    for g in range(n):
        if os.path.exists(f"worker_{tag}_{g}.log"):
            print(open(f"worker_{tag}_{g}.log").read())

def name(lr, seed):
    return "lltTrainingLoop-t2-lr" + format(lr, ".0e").replace("e-0", "e-") + "-s" + str(seed)

# Sanity run first: 30 steps. If it fails, nothing else starts.
run_pool([{"runId": "lltTrainingLoop-t2-sanity", "label": "sanity", "cfg": dict(T2, lr=1e-3, seed=0, steps=30, eval_every=30)}], "sanity")
sanity = json.load(open("runs/lltTrainingLoop-t2-sanity.json"))
assert sanity["status"] == "ok", sanity["status"]
print("sanity ok, final val", sanity["evals"]["finalValLoss"], "tok/s", sanity["tokPerSecInclEval"])


In [ ]:

LRS = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
jobs = [{"runId": name(lr, s), "label": "ablation:lr=" + format(lr, ".0e"), "cfg": dict(T2, lr=lr, seed=s)} for s in range(3) for lr in LRS]
run_pool(jobs, "sweep")
res = {}
for j in jobs:
    p = "runs/" + j["runId"] + ".json"
    if os.path.exists(p):
        r = json.load(open(p))
        res.setdefault(j["cfg"]["lr"], []).append(r["evals"]["finalValLoss"] if r["evals"]["finalValLoss"] is not None else float("nan"))
for lr in LRS:
    print(lr, res.get(lr))
pool = [lr for lr in res if all(v == v for v in res[lr])]
best = min(pool, key=lambda lr: sum(res[lr]) / len(res[lr])) if pool else None
print("best lr", best, "elapsed min", round((time.time() - START) / 60, 1))


In [ ]:

extra = [{"runId": name(best, s), "label": "seeds:lr=" + format(best, ".0e"), "cfg": dict(T2, lr=best, seed=s)} for s in range(3, 10)] if best else []
if extra:
    run_pool(extra, "seeds")
vals = []
for s in range(10):
    p = "runs/" + name(best, s) + ".json" if best else ""
    if best and os.path.exists(p):
        vals.append(json.load(open(p))["evals"]["finalValLoss"])
print("best lr", best, "seeds", len(vals), "final val", vals)
summary = {"best": best, "sweep": {str(k): v for k, v in res.items()}, "bestSeeds": vals,
           "sessionMinutes": round((time.time() - START) / 60, 1)}
json.dump(summary, open("runs/lltTrainingLoop-t2-summary.json", "w"), indent=1)
print(summary)
